In [1]:
try:
    get_ipython().run_line_magic("matplotlib", "qt")
except NameError:
    import matplotlib
    matplotlib.use("QtAgg")

In [2]:
from typing import Any

from bloqade import squin
from bloqade.gemini.common.dialects import qubit
from bloqade.lanes.arch.gemini.logical import steane7_initialize
from bloqade.lanes.arch.gemini.physical import get_arch_spec
from bloqade.lanes.upstream import squin_to_move
from bloqade.lanes.heuristics.physical import (
    PhysicalLayoutHeuristicGraphPartitionCenterOut,
    PhysicalPlacementStrategy,
)
from bloqade.lanes.visualize import debugger
from bloqade.types import Qubit
from kirin.dialects import ilist
from kirin.dialects.ilist import IList

kernel = squin.kernel.add(qubit)
kernel.run_pass = squin.kernel.run_pass


def slot_allocator():
    # canonical slot order
    slot_words = ilist.IList([0, 4, 8, 12, 16, 2, 6, 10, 14, 18])

    slots = IList(
        [
            IList([(0, word_id, site_id) for site_id in range(7)])
            for word_id in slot_words
        ]
    )

    @kernel
    def qalloc_slot(
        slot_index: int, theta: float, phi: float, lam: float
    ) -> IList[Qubit, Any]:
        def allocate_at(address: tuple[int, int, int]):
            return qubit.new_at(address[0], address[1], address[2])

        addresses = slots[slot_index]

        reg = ilist.map(allocate_at, addresses)

        steane7_initialize(theta, phi, lam, reg)

        return reg

    @kernel
    def qalloc(
        slot_indices: list[int] | IList[int, Any],
        theta: float = 0.0,
        phi: float = 0.0,
        lam: float = 0.0,
    ) -> IList[IList[Qubit, Any], Any]:

        def _inner(slot_index: int):
            return qalloc_slot(slot_index, theta, phi, lam)

        return ilist.map(_inner, slot_indices)

    return qalloc, qalloc_slot


qalloc, qalloc_slot = slot_allocator()


@kernel(typeinfer=True)
def main():
    reg = qalloc([0, 1], 0.0, 0.0, 0.0)


move_mt = squin_to_move(
    main,
    PhysicalLayoutHeuristicGraphPartitionCenterOut(),
    PhysicalPlacementStrategy(),
    logical_initialize=False,
    no_raise=False,
)
move_mt.verify()
move_mt.print()
debugger(move_mt, get_arch_spec())

func.func @main() -> !Bottom {
  ^0(%main_self):
  │       %0 = lanes.move.load() : !py.State
  │       %1 = lanes.move.fill(current_state=%0){location_addresses=(0x0000000000000000, 0x0000000001000000, 0x0000000002000000, 0x0000000003000000, 0x0000000004000000, 0x0000000005000000, 0x0000000006000000, 0x0000040000000000, 0x0000040001000000, 0x0000040002000000, 0x0000040003000000, 0x0000040004000000, 0x0000040005000000, 0x0000040006000000) : !py.tuple[!py.LocationAddress, ...]} : !py.State
  │            lanes.move.store(current_state=%1)
  │       %2 = py.constant.constant 'Begin Steane7 Initialize' : !py.str
  │   %theta = py.constant.constant 0.0 : !py.float
  │   %angle = py.constant.constant 0.25 : !py.float
  │ %angle_1 = py.constant.constant -0.25 : !py.float
  │ %angle_2 = py.constant.constant 0.5 : !py.float
  │       %3 = py.constant.constant 'End Steane7 Initialize' : !py.str
  │            debug.info(msg=%2, inputs=())
  │       %4 = lanes.move.local_rz(current_state=%1, rot

## Gemini logical dialect version

This version lets the Gemini logical pipeline insert the logical initialization. `new_at(0, word, 0)` pins each logical qubit to a logical home site; `transversal_rewrite=True` then expands each logical site to its seven Steane physical sites.

In [5]:
from bloqade import squin
from bloqade.gemini import logical as gemini_logical
from bloqade.gemini.common.dialects.qubit import new_at
from bloqade.lanes.dialects import move
from bloqade.lanes.logical_mvp import (
    compile_squin_to_move as compile_logical_squin_to_move,
)
from kirin.dialects import ilist


@gemini_logical.kernel(aggressive_unroll=True)
def logical_pinned_main():
    q0 = new_at(0, 0, 0)  # logical home site: zone 0, word 0, site 0
    q1 = new_at(0, 4, 0)  # logical home site: zone 0, word 4, site 0
    reg = ilist.IList([q0, q1])

    squin.cx(reg[0], reg[1])
    gemini_logical.terminal_measure(reg)


logical_move_mt = compile_logical_squin_to_move(
    logical_pinned_main,
    transversal_rewrite=False,
    insert_return_moves=False,
    no_raise=False,
)

physical_move_mt = compile_logical_squin_to_move(
    logical_pinned_main,
    transversal_rewrite=True,
    insert_return_moves=False,
    no_raise=False,
)

print("Logical pipeline output: compiler-inserted logical initialization")
for stmt in logical_move_mt.callable_region.walk():
    if isinstance(stmt, move.LogicalInitialize):
        print("logical home sites:", stmt.location_addresses)

print("\nAfter transversal rewrite: logical sites expanded to physical sites")
for stmt in physical_move_mt.callable_region.walk():
    if isinstance(stmt, move.PhysicalInitialize):
        for logical_index, addresses in enumerate(stmt.location_addresses):
            print(f"logical qubit {logical_index} physical sites:", addresses)

# Use this if you want to inspect the full expanded move IR.
# physical_move_mt.print()
debugger(physical_move_mt, get_arch_spec())


Logical pipeline output: compiler-inserted logical initialization
logical home sites: (0x0000000000000000, 0x0000040000000000)

After transversal rewrite: logical sites expanded to physical sites
logical qubit 0 physical sites: (0x0000000000000000, 0x0000000001000000, 0x0000000002000000, 0x0000000003000000, 0x0000000004000000, 0x0000000005000000, 0x0000000006000000)
logical qubit 1 physical sites: (0x0000040000000000, 0x0000040001000000, 0x0000040002000000, 0x0000040003000000, 0x0000040004000000, 0x0000040005000000, 0x0000040006000000)
